# Minimum dominating set

All fifteen code stages: fresh six-node graph (edge probability 0.6, seed 8), six decision bits, fourteen encoded slack bits, six QAOA layers, penalty 10, 70 optimizer iterations, 70% lower-tail CVaR, optimized and uniform samples, and an independent exhaustive constrained solution.

Reuse the native qfunc for the original seed-8 closed neighborhoods and retain every slack register. Exact local CVaR integrates all joint decision/slack masses. Native Nelder–Mead and an explicit linear pi schedule replace the unpinned hosted optimizer; actual query counts are shown separately from the iteration budget. Use 1000 actual CPU draws for each comparison. NetworkX seeded spring layout renders the same underlying graph with selected nodes and incident edges; it does not reproduce hosted layout coordinates. Exhaustive search over 64 decision assignments replaces Couenne and only runs after sampling. No historical optimizer trajectory or optimum guarantee for the observed quantum sample.

## Set up the model and plotting tools

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.minimum_dominating_set.overview.model import (
    fresh_model,
)
from examples.synthesis.applications.optimization.minimum_dominating_set.overview.plots import (
    draw_histogram,
    draw_nodes,
    draw_trace,
)
from examples.synthesis.applications.optimization.minimum_dominating_set.overview.workflow import (
    INITIAL,
    SHOTS,
    build_program,
    run_workflow,
)


## Define the domination condition

In [ ]:
model = fresh_model()
print("Every closed neighborhood must intersect the chosen set.")
print("Closed neighborhoods:", model.neighborhoods)


## Generate and draw the original graph

In [ ]:
artifact_dir = Path("artifacts/development/minimum-dominating-set")
display(
    Image(
        filename=str(
            draw_nodes(
                model, artifact_dir / "source-graph.png", "Original six-node graph"
            )
        )
    )
)


## Inspect the constraint encoding

In [ ]:
for neighborhood, coefficients in zip(
    model.neighborhoods, model.slack_weights, strict=True
):
    print(f"sum of choices in {neighborhood} >= 1; slack weights {coefficients}")


## Set the original QAOA workflow dimensions

In [ ]:
print("Six layers, penalty 10, 70 iterations, 70% lower-tail cost")
print("Starting angles:", INITIAL)


## Synthesize the initial native program

In [ ]:
starting_program = build_program(INITIAL)
print(starting_program.report["resources"])


## Optimize all six layers

In [ ]:
result = run_workflow()
print("Actual objective evaluations:", len(result.trace))
print("Converged:", result.optimizer.converged)
print(
    "Initial and selected CVaR:",
    result.trace[0].cost,
    min(r.cost for r in result.trace),
)


## Read the actual cost trace

In [ ]:
display(Image(filename=str(draw_trace(result, artifact_dir / "cost-trace.png"))))


## Inspect the five lowest observed penalized costs

In [ ]:
for row in result.optimized[:5]:
    print(row)


## Check the uniform comparison draws

In [ ]:
print(
    "Uniform and optimized draws:",
    sum(row.count for row in result.uniform),
    sum(row.count for row in result.optimized),
)
print("Configured draws per comparison:", SHOTS)


## Compare the observed distributions

In [ ]:
display(
    Image(filename=str(draw_histogram(result, artifact_dir / "cost-histogram.png")))
)


## Select the best actually observed joint assignment

In [ ]:
best = result.optimized[0]
print("Best actually observed:", best)


## Draw the observed set

In [ ]:
display(
    Image(
        filename=str(
            draw_nodes(
                model,
                artifact_dir / "observed-set.png",
                f"Observed cost {best.cost}, feasible={best.feasible}",
                best.decisions,
            )
        )
    )
)


## Read the independent exact constrained solution

In [ ]:
print("All exact minimum decision words:", result.classical)
print("Minimum set size:", result.classical[0].bit_count())


## Draw one exact set

In [ ]:
display(
    Image(
        filename=str(
            draw_nodes(
                model,
                artifact_dir / "exact-set.png",
                "One exact minimum dominating set",
                result.classical[0],
            )
        )
    )
)
